In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "NOTEBOOKS":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "DATA"
RAW_DIR = DATA_DIR / "RAW"
PROCESSED_DIR = DATA_DIR / "PROCESSED"
MODELS_DIR = PROJECT_ROOT / "MODELS"
OUTPUTS_DIR = PROJECT_ROOT / "OUTPUTS"
PLOTS_DIR = OUTPUTS_DIR / "PLOTS"

for d in [PROCESSED_DIR, MODELS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)

from sklearn.model_selection import train_test_split

candidates = list(RAW_DIR.rglob("*.csv"))
preferred = [p for p in candidates if "ai4i" in p.name.lower() or "predictive" in p.name.lower()]
DATA_PATH = preferred[0] if preferred else candidates[0]
df = pd.read_csv(DATA_PATH)

df.columns = [c.strip().lower().replace(" ", "_").replace("-", "_") for c in df.columns]
target = "machine_failure"
if target not in df.columns:
    raise KeyError(f"Expected target '{target}'. Found: {df.columns.tolist()}")

print("Initial shape:", df.shape)
print("Missing values before treatment:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

Project root: C:\Users\sitar\Downloads\ASSIGNMENT_CODE\PREDICTIVE-MAINTENANCE
Python: 3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]
Initial shape: (10000, 14)
Missing values before treatment: 0
Duplicate rows: 0


In [2]:
# Remove exact duplicates
df = df.drop_duplicates().copy()

# Separate target
X = df.drop(columns=[target]).copy()
y = df[target].astype(int).copy()

# Drop identifier-like columns when present
id_cols = [c for c in X.columns if c in ["udi", "product_id"] or c.endswith("_id")]
X = X.drop(columns=id_cols, errors="ignore")

# Encode categorical machine type
if "type" in X.columns:
    X["type"] = X["type"].map({"L": 0, "M": 1, "H": 2}).fillna(X["type"].astype("category").cat.codes)

numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

# Train/test split before fitting imputers/outlier limits.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Median imputation based only on training data
train_medians = X_train[numeric_cols].median()
X_train[numeric_cols] = X_train[numeric_cols].fillna(train_medians)
X_test[numeric_cols] = X_test[numeric_cols].fillna(train_medians)

print("Missing values after imputation — train:", int(X_train.isna().sum().sum()))
print("Missing values after imputation — test:", int(X_test.isna().sum().sum()))

Missing values after imputation — train: 0
Missing values after imputation — test: 0


In [3]:
# IQR-based outlier capping.
# We cap extreme values rather than deleting rows, preserving rare failure examples.
outlier_limits = {}
for col in numeric_cols:
    q1 = X_train[col].quantile(0.25)
    q3 = X_train[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_limits[col] = (lower, upper)
    X_train[col] = X_train[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)

print("Outlier treatment: IQR capping applied to numerical features.")
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

# Persist processed splits for later notebooks.
train_processed = X_train.copy()
train_processed[target] = y_train.values
test_processed = X_test.copy()
test_processed[target] = y_test.values

train_processed.to_csv(PROCESSED_DIR / "train_processed.csv", index=False)
test_processed.to_csv(PROCESSED_DIR / "test_processed.csv", index=False)
print("Saved processed datasets.")

Outlier treatment: IQR capping applied to numerical features.
Training shape: (8000, 11)
Testing shape: (2000, 11)
Saved processed datasets.
